[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/20_capstone.ipynb)

# Capstone — an end-to-end fine-tune: data → train → evaluate → deploy

> **Fine-tuning series — 20 of 26.** The whole pipeline on one small real dataset, chaining the methods from earlier notebooks. Self-contained: run **Setup**, then the rest.

## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos d\u00edas."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' \u2014 a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## The plan

We'll build a small **instruction-following assistant** from a real dataset and take it
all the way to a saved, runnable model — each step links back to the notebook that teaches it:

1. **Prepare data** (→ 02c) — load a real Hub dataset, clean, split, format.
2. **Smoke test** (→ 02b) — prove the pipeline can learn before a long run.
3. **Train** (→ 04 / 08) — LoRA SFT with loss on the response only.
4. **Evaluate** (→ 19) — perplexity + generations, base vs. tuned.
5. **Align** (→ 10) — an optional short DPO pass.
6. **Save & deploy** (→ 17) — adapter out, reload, generate.

On a GPU you'd swap step 3 for **QLoRA + Unsloth** (notebooks 05 / 07); the data, eval,
and deploy steps are identical.

### 1. Prepare data (→ 02c)

Load a small slice of the real **Alpaca** instruction dataset, turn each row into a chat
conversation, drop empties, and split train / val / test.

In [ ]:
from datasets import load_dataset

raw = load_dataset("tatsu-lab/alpaca", split="train[:80]")   # small slice for a fast demo

def to_messages(ex):
    user = ex["instruction"] + (("\n" + ex["input"]) if ex["input"].strip() else "")
    return {"messages": [{"role": "user", "content": user},
                         {"role": "assistant", "content": ex["output"]}]}

ds = raw.map(to_messages, remove_columns=raw.column_names)
ds = ds.filter(lambda e: len(e["messages"][1]["content"].strip()) > 0)
split = ds.train_test_split(test_size=0.2, seed=42)
hold = split["test"].train_test_split(test_size=0.5, seed=42)
train_ds, val_ds, test_ds = split["train"], hold["train"], hold["test"]
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")
print(train_ds[0]["messages"])

Format to **response-only** training features (loss masked to the assistant turn — the
lesson from notebook 08), capped to a max length so long answers don't blow up memory.

In [ ]:
MAXLEN = 384

def mask_to_response(ex):
    msgs = ex["messages"]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, return_dict=True)["input_ids"][:MAXLEN]
    prompt_len = min(len(tokenizer.apply_chat_template(
        msgs[:1], tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"]), len(ids))
    labels = [-100] * prompt_len + ids[prompt_len:]
    return {"input_ids": ids, "labels": labels}

train_feat = train_ds.map(mask_to_response, remove_columns=train_ds.column_names)
print("loss is computed only on:",
      repr(tokenizer.decode([t for t in train_feat[0]["labels"] if t != -100])[:80]))

### 2. Smoke test (→ 02b)

Before the real run, confirm the pipeline can memorize a handful of examples — loss should
crash toward 0. If it can't, the model/data are fine but the *pipeline* is broken.

In [ ]:
from transformers import Trainer, TrainingArguments
from peft import get_peft_model

def pad_collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        n = maxlen - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * n)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * n)
        out["labels"].append(b["labels"] + [-100] * n)
    return {k: torch.tensor(v) for k, v in out.items()}

tiny = Dataset.from_list([train_feat[i] for i in range(min(8, len(train_feat)))])
sm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
sm = get_peft_model(sm, lora_cfg)
sm_tr = Trainer(model=sm, train_dataset=tiny, data_collator=pad_collate,
    args=TrainingArguments(output_dir="cap_smoke", per_device_train_batch_size=2,
        max_steps=20, learning_rate=2e-4, logging_steps=10, bf16=BF16_OK, report_to="none"))
sm.print_trainable_parameters()
h = sm_tr.train()
print("smoke loss ->", round(h.training_loss, 3), "(should be trending toward 0)")
del sm, sm_tr; cleanup()

### 3. Train (→ 04 / 08)

The real run: LoRA SFT on the full train split.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainer = Trainer(model=model, train_dataset=train_feat, data_collator=pad_collate,
    args=TrainingArguments(output_dir="capstone_sft", per_device_train_batch_size=2,
        max_steps=40, learning_rate=2e-4, logging_steps=10, bf16=BF16_OK, report_to="none"))
trainer.train()

# Save just the adapter, then free the training model — eval reloads it on a single
# base (adapter on top), so we never hold two full models in memory at once.
trainer.save_model("capstone_adapter")
tokenizer.save_pretrained("capstone_adapter")
del trainer, model; cleanup()
print("adapter saved -> capstone_adapter")

### 4. Evaluate (→ 19)

Perplexity and generations, base vs. tuned, on the held-out **test** split.

In [ ]:
import math

@torch.no_grad()
def perplexity(model, texts):
    losses = [model(**tokenizer(t, return_tensors="pt").to(model.device),
                    labels=tokenizer(t, return_tensors="pt").to(model.device)["input_ids"]).loss.item()
              for t in texts]
    return math.exp(sum(losses) / len(losses))

@torch.no_grad()
def answer(model, prompt, max_new_tokens=50):
    enc = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
            add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

from peft import PeftModel
test_texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in test_ds]
q = test_ds[0]["messages"][0]["content"]

# One base model in memory; measure it, then add the adapter on top for the tuned pass.
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device).eval()
base_ppl, base_ans = perplexity(base, test_texts), answer(base, q)

tuned = PeftModel.from_pretrained(base, "capstone_adapter").eval()
tuned_ppl, tuned_ans = perplexity(tuned, test_texts), answer(tuned, q)

print(f"perplexity   base {base_ppl:6.2f}  ->  tuned {tuned_ppl:6.2f}   (lower = better)")
print("\nQ    :", q[:90])
print("base :", base_ans)
print("tuned:", tuned_ans)
del tuned, base; cleanup()

### 5. Align with DPO (→ 10) — optional

A short preference-optimization pass. In a real project you'd collect `{prompt, chosen,
rejected}` triples (often by sampling your SFT model and ranking the outputs); here we
reuse the tiny inline `preferences` just to run the step. Set `RUN_DPO = True` to execute
(it adds a couple of minutes on CPU).

In [ ]:
RUN_DPO = False   # flip to True to run the alignment pass

if RUN_DPO:
    from trl import DPOTrainer, DPOConfig
    dpo = DPOTrainer(
        model=MODEL_ID,
        args=DPOConfig(output_dir="capstone_dpo", per_device_train_batch_size=1,
            gradient_accumulation_steps=4, max_steps=8, learning_rate=5e-6, beta=0.1,
            max_length=512, bf16=BF16_OK, fp16=FP16_OK, logging_steps=2, report_to="none"),
        train_dataset=pref_ds, peft_config=lora_cfg)
    dpo.train(); del dpo; cleanup()
    print("DPO pass done")
else:
    print("skipped (RUN_DPO = False); see notebook 10 for the full DPO walkthrough")

### 6. Save & deploy (→ 17)

The adapter was saved right after training (a few MB). Here we reload it on top of the base
and generate — proving the artifact works after a round-trip. To ship: `merge_and_unload()`
for zero-overhead serving, then export to the Hub / GGUF (Ollama, llama.cpp) / vLLM (notebook 17).

In [ ]:
import os
print("adapter files (a few MB, hot-swappable on one base):", os.listdir("capstone_adapter"))

# Load base + adapter fresh and serve a prompt — proves the saved artifact works.
reloaded = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device),
    "capstone_adapter").eval()
print("deployed answer ->", answer(reloaded, q))
del reloaded; cleanup()
# To ship: merged = reloaded.merge_and_unload() for zero-overhead serving, then export
# to the Hub / GGUF (Ollama, llama.cpp) / vLLM — see notebook 17.

### Recap — the whole pipeline

| Step | This capstone | Taught in |
|---|---|---|
| Prepare data | load Alpaca → chat → clean → split | **02c** |
| Smoke test | overfit 8 examples | **02b** |
| Train | LoRA SFT, response-only loss | **04**, **08** |
| Evaluate | perplexity + generations, base vs. tuned | **19** |
| Align | optional DPO | **10** |
| Deploy | save adapter → reload → generate | **17** |

That's a complete fine-tune. Swap the dataset for your own (notebook 02c), scale the model
with QLoRA + Unsloth on a GPU (05 / 07), and re-run the same six steps.